<a href="https://colab.research.google.com/github/Laiba-saeed92/Deep-Learning-and-NLP-Projects/blob/main/Question_Answering_using_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [ ]:
!pip install transformers torch datasets nltk


In [ ]:
import transformers, torch, nltk

In [ ]:
print(transformers.__version__)
print(torch.__version__)
print(nltk.__version__)

4.57.1
2.8.0+cu126
3.9.1


In [ ]:
!pip uninstall -y pyarrow pandas datasets
!pip install --no-cache-dir "pyarrow==15.0.2" "pandas==2.2.2" "datasets==3.0.1"


Found existing installation: pyarrow 21.0.0
Uninstalling pyarrow-21.0.0:
  Successfully uninstalled pyarrow-21.0.0
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: datasets 4.2.0
Uninstalling datasets-4.2.0:
  Successfully uninstalled datasets-4.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 227.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 381.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 314.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 148.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: fsspec
 

In [ ]:
import pyarrow, pandas, datasets
print("pyarrow:", pyarrow.__version__)
print("pandas:", pandas.__version__)
print("datasets:", datasets.__version__)


pyarrow: 15.0.2
pandas: 2.2.2
datasets: 3.0.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from datasets import load_dataset


In [ ]:
dataset=load_dataset("squad")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [ ]:
print(dataset["train"][0])

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [ ]:
#Load BERT tokenizer and model
tokenizer=AutoTokenizer.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")
model=AutoModelForQuestionAnswering.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
#Simple preprocessing (adds dummy labels so Trainer won't fail)
def preprocess_input(examples):
    encodings = tokenizer(
        examples["question"], examples["context"],
        truncation=True, padding="max_length", max_length=384
    )
    # Add dummy positions (so model has labels)
    encodings["start_positions"] = [0] * len(examples["question"])
    encodings["end_positions"] = [0] * len(examples["question"])
    return encodings

tokenized_dataset = dataset.map(preprocess_input, batched=True)


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
#Prepare small subset (to make training fast)
small_train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(10000))
small_eval_dataset=tokenized_dataset["validation"].shuffle(seed=42).select(range(200))




In [ ]:
training_args=TrainingArguments(
    output_dir="./bert-qa-project",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",

)

In [ ]:
trainer= Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset= small_train_dataset,
    eval_dataset= small_eval_dataset,
)

/tmp/ipython-input-1015105053.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer= Trainer(


In [ ]:
#Train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.010700,0.000002
2,0.000000,0.000000
3,0.000000,0.000000


In [ ]:
#Save the fine-tuned model
trainer.save_model("./bert-qa-finetuned")
tokenizer.save_pretrained("./bert-qa-finetuned")


('./bert-qa-finetuned/tokenizer_config.json',
 './bert-qa-finetuned/special_tokens_map.json',
 './bert-qa-finetuned/vocab.txt',
 './bert-qa-finetuned/added_tokens.json',
 './bert-qa-finetuned/tokenizer.json')

In [ ]:
# Test the fine-tuned model

'''The pipeline("question-answering") handles all of that automatically:
It encodes the question and context together in the correct format.
It runs the model.
It decodes the predicted span into readable text'''
#pipeline("question-answering") simplifies encoding & decoding
from transformers import pipeline
qa_pipeline= pipeline("question-answering", model="./bert-qa-finetuned", tokenizer="./bert-qa-finetuned")
context="laiba lives in islamabad pakistan. She studies computer system engineering from uet peshawar. She is 22 years old and a hardworking girl. Laiba's sister name is royala . Royala is a banker"
question="what does she study?"
result=qa_pipeline(context=context, question=question)
print(result)


Device set to use cuda:0


{'score': 1.9741716941745757e-11, 'start': 0, 'end': 5, 'answer': 'laiba'}
